In [19]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd

In [20]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

/tmp/ipykernel_12471/3003301750.py:2: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [21]:
# === Global Configuration and Constants ===
start_sqrt_s = 101  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

lst_sigma_tot_born = []
lst_sqrt_s = []
lst_error = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0732

max_sqrt_s = 13000
step = 100
n_points = 10000

model_params = {
    'atlas': {
        'pl':  {'mg': 0.417, 'a1': 1.563, 'a2': 2.22}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}

lst_amp_born = []
lst_sqrt_s = []
lst_sigma_tot_born = []




In [22]:
# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


# -------------------------------
# Inner integral (over phi)
# -------------------------------
def phi_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) - 
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    result, _ = fixed_quad(integrand, 0, 2*np.pi, n=n_points)
    return result

# -------------------------------
# Outer integral (over k)
# -------------------------------
def k_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    return phi_integral(k, mg, a1, a2, m2_func, q, n_points)

# -------------------------------
# Double integral computation
# -------------------------------
def compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points=10000):
    result, _ = fixed_quad(
        lambda k: k_integral(k, mg, a1, a2, m2_func, q, n_points),
        0, sqrt_s_val, 
        n=n_points
    )
    return result

def born_amp(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot_born(amp_born_value, s):
    return amp_born_value.imag / s * 0.389379323


In [23]:

def calculate_born_cross_sections(start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points=10000):
    """Calculate cross sections for all sqrt_s values"""
    
    # Generate array of sqrt_s values
    sqrt_s_values = np.arange(start_sqrt_s, max_sqrt_s + step, step)
    
    # Process each sqrt_s value
    lst_sigma_tot_born = []
    lst_sqrt_s = []
    lst_amp_born = []
    
    for sqrt_s_val in sqrt_s_values:
        # Compute the double integral
        q = 0  # Assuming q=0 as in the original code
        diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points)
        
        # Calculate amplitude and cross section
        s = sqrt_s_val * sqrt_s_val
        amp_born_value = born_amp(diff_T, s, epsilon, 0)
        sigma_tot_born_value = sigma_tot_born(amp_born_value, s)
        
        # Store results
        lst_sigma_tot_born.append(sigma_tot_born_value)
        lst_sqrt_s.append(sqrt_s_val)
        lst_amp_born.append(amp_born_value)
    
    return lst_sigma_tot_born, lst_sqrt_s, lst_amp_born



In [24]:
mass_model = 'pl'
ensemble = 'atlas'
m2_func = get_m2_function(mass_model)
params = model_params[ensemble][mass_model]
mg, a1, a2 = params['mg'], params['a1'], params['a2']
epsilon = epsilon_values[ensemble]

# Calculate all cross sections
lst_sigma_tot_born, lst_sqrt_s, lst_amp_born = calculate_born_cross_sections(
    start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points
)

lst_s = [val**2 for val in lst_sqrt_s]

In [ ]:
n = 1000
q_max = 0.1


# -------------------------------
# Full chi(s,b) including k, phi, and new q integral
# -------------------------------
def chi_integral(sqrt_s_values, mg, a1, a2, m2_func, epsilon, b):
    """
    Computes chi(s,b) = (1/s) ∫_0^qmax q dq J0(b q) [i 8 s^(1+ε) (∫_0^√s k dk ∫_0^2π dφ (T1-T2)) ]
    """
    chi_list = []

    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        # integrand over q
        def q_integral(q):

            # compute existing double integral over k and phi
            diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q)
            return (q * j0(b * q) * born_amp(diff_T, s, epsilon, -(q**2)))/s

        # integrate real and imaginary parts separately
        real_part, _ = quad(lambda q: q_integral(q).real, 0, q_max, limit=n, epsabs=1e-10, epsrel=1e-10)
        imag_part, _ = quad(lambda q: q_integral(q).imag, 0, q_max, limit=n, epsabs=1e-10, epsrel=1e-10)

        chi_list.append((real_part + 1j * imag_part))

    return chi_list

amp_list = []
b_max = 10

# -------------------------------
# Eikonal amplitude A_eik(s,t)
# -------------------------------
def eikonal_amplitude(sqrt_s_values, mg, a1, a2, m2_func, epsilon):

    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2
    
        # integrand over b
        def b_integrand(b):
            chi_val = chi_integral([sqrt_s_val], mg, a1, a2, m2_func, epsilon, b)[0]
            return b * (1 - np.exp(1j * chi_val))
        
        # integrate real and imaginary parts separately
        real_part, _ = quad(lambda b: b_integrand(b).real, 0, b_max, limit=n, epsabs=1e-10, epsrel=1e-10)
        imag_part, _ = quad(lambda b: b_integrand(b).imag, 0, b_max, limit=n, epsabs=1e-10, epsrel=1e-10)

        A_eik = 1j * s * (real_part + 1j * imag_part)
        amp_list.append(A_eik)

    return amp_list


# Example usage
A_eik_values = eikonal_amplitude(lst_sqrt_s, mg, a1, a2, m2_func, epsilon)
# print(A_eik_values)



In [26]:
def sigma_tot_eik(amp, s):
    return (4*np.pi)/s * amp.imag * 0.389379323   

lst_sigma_tot_eik = [sigma_tot_eik(amp, s) for amp, s in zip(A_eik_values, lst_s)]
print(lst_sigma_tot_eik)


[111.06595016590539, 119.22115800062417, 124.13583888343712, 127.67867662779128, 130.4552382474967, 132.74063996328084, 134.68369812191952, 136.3741971536781, 137.87054959471442, 139.21292467731175, 140.43014771743478, 141.54361416301631, 142.56964684298566, 143.52098559963164, 144.40776667606065, 145.23818837611987, 146.01897646281805, 146.75571756628383, 147.45310315675192, 148.11511143440737, 148.74514519447308, 149.3461378764279, 149.92063622774512, 150.4708655147193, 150.99878152706424, 151.50611246357255, 151.99439297509483, 152.46499206489307, 152.9191361311864, 153.35792813353305, 153.78236364065577, 154.19334434990418, 154.5916895421365, 154.97814583943762, 155.35339555895644, 155.7180638986385, 156.07272514566242, 156.4179080629687, 156.75410058117393, 157.0817539007276, 157.40128609114532, 157.71308525958622, 158.01751234920175, 158.3149036180153, 158.6055728411461, 158.8898132726441, 159.16789939776763, 159.44008850202061, 159.7066220794832, 159.9677270998044, 160.223617150

In [27]:
import re


def add_iterative_curve(fig, x_data, y_data, 
                        curve_name:str=None, color:str='blue', line_type:str='lines+markers'):

    fig.add_trace(go.Scatter(
    x = x_data,
    y = y_data,
    mode=line_type, 
    name=curve_name,
    line=dict(
        color=color,
        width=2),
    marker=dict(size=4))
)
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

fig_amp = go.Figure()

add_iterative_curve(fig_amp, lst_sqrt_s, np.imag(lst_amp_born), curve_name='Im(A born)', color='blue')
add_iterative_curve(fig_amp, lst_sqrt_s, np.imag(A_eik_values), curve_name='Im(A eikonal)', color='red')
fig_amp.update_layout(
    title='Amplitude Im vs. sqrt(s)',
    xaxis=dict(
        title='sqrt(s) [GeV]',
        type='log',
    ),
    yaxis=dict(
        title='Amp',
    ),
    showlegend=True,
    legend=dict(
        title='Model/Data'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
fig_amp.show(renderer = 'browser')


fig_sigma = go.Figure()

add_iterative_curve(fig_sigma, lst_sqrt_s, lst_sigma_tot_born, curve_name='sigma tot born')
add_iterative_curve(fig_sigma, lst_sqrt_s, lst_sigma_tot_eik, curve_name='sigma tot eikonal', color='red')


# Add ATLAS data
fig_sigma.add_trace(go.Scatter(
    x=x_atlas,
    y=y_atlas,
    mode='markers',
    marker=dict(
        color='black',
        size=6,
        symbol='square'
    ),
    error_y=dict(
        type='data',
        array=y_error_atlas,
        visible=True
    ),
    name='ATLAS Data'
))

# Configure layout
fig_sigma.update_layout(
    title='Sigma Tot vs. sqrt(s)',
    xaxis=dict(
        title='sqrt(s) [GeV]',
        type='log',
    ),
    yaxis=dict(
        title='Sigma Tot [mb]',
    ),
    showlegend=True,
    legend=dict(
        title='Model/Data'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_sigma.update_xaxes(gridcolor='lightgray')
fig_sigma.update_yaxes(gridcolor='lightgray')

fig_sigma.show(renderer = 'browser')

Opening in existing browser session.
Opening in existing browser session.
